import sys

from pathlib import Path

sys.path.append(str(Path.cwd().parent/'src'))

In [2]:
from pdf_ingestion import PdfIngestion

In [3]:
ingestor = PdfIngestion(chunk_size=1200,chunk_overlap=180)

pdf_chunks = ingestor.process('C:\Projets_rag_personnels\data\pdf_files\Capstone_FinalReport.pdf')


print(f'Total of chunks : {len(pdf_chunks)}')

for i, chunk in enumerate(pdf_chunks[18:21]):

  print(f'CHUNK {i+1}')
  print(chunk.page_content)
  print(f'Source :page {chunk.metadata.get('page')}')

<>:3: SyntaxWarning: invalid escape sequence '\P'
<>:3: SyntaxWarning: invalid escape sequence '\P'
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_27008\2859246122.py:3: SyntaxWarning: invalid escape sequence '\P'
  pdf_chunks = ingestor.process('C:\Projets_rag_personnels\data\pdf_files\Capstone_FinalReport.pdf')


Consider using the pymupdf_layout package for a greatly improved page layout analysis.
Total of chunks : 76
CHUNK 1
. Early warning systems are designed to address this problem through the early detection of declining customer activity using behavioural indicators (Verbeke et al., 2012). This is very useful in the logistics sector because the volume of shipments is likely to be indicative of the customer’s demand patterns.
Source :page 7
CHUNK 2
9 2.5 Customer Segmentation and Portfolio Analysis Customer segmentation is the process of grouping customers according to their characteristics and behaviours (Wedel and Kamakura, 2000). This is very useful in developing appropriate marketing strategies. Portfolio analysis is another useful concept that helps firms segment their customers using the BCG matrix (Henderson, 1970). This matrix is useful in developing appropriate marketing strategies according to the performance of the customer segments. This is very useful in allocating resources 

In [4]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from dotenv import load_dotenv
import os

load_dotenv()
print(os.getenv('OPENAI_API_KEY')[:10] + '...')





embed = OpenAIEmbeddings(model='text-embedding-3-small')

pdf_vector_store = Chroma(collection_name='Capstone_focused_docs',
                          embedding_function=embed,
                          persist_directory='/content/drive/MyDrive/Cour de RAG/mes _projets RAG/vector_store_new')

pdf_vector_store.add_documents(pdf_chunks)

retriever = pdf_vector_store.as_retriever(search_kwargs = {'k':3})


# let us test our retriever

sample_search = retriever.invoke("tell me about the analysis of revenue concentration")

for i, result in enumerate(sample_search[:3]):

  print(f'RESULT {i+1}')

  print(result.page_content,'\n')

sk-proj-rR...


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_27008\939186484.py:16: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  pdf_vector_store = Chroma(collection_name='Capstone_focused_docs',


RESULT 1
18 opportunities, while others may represent growth potential through improved targeting and pricing strategies. From a strategic perspective, high-revenue regions such as the Northwest act as critical revenue anchors but also represent key sources of risk. At the same time, lower-performing regions may offer opportunities for expansion. Finally, the large “Unknown” geographic category reduces the precision of the analysis and should be addressed as a data quality priority. 4.1.6 Strategic Implications for Revenue Structure The results of Section 4.1 provide a critical foundation for interpreting customer churn dynamics and revenue risk. First, the extreme level of customer concentration indicates that revenue is driven by a small number of high-value accounts. The company shows strong dependency on approximately 1,000 “whale” clients, with the top 5% generating around 74% of total revenue. Second, sector concentration suggests that industry-specific dynamics will play a key r

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser


llm = ChatOpenAI(model = 'gpt-4o-mini')

parser = StrOutputParser()

prompt = ChatPromptTemplate.from_template ("""
                                           
You are a technical assistant for our data analytics team.
Answer the question below focusing on the context below.
If there is no answer in the context, just say: "there is no answer"


QUESTION:
{question}


CONTEXT:
{context}


ANSWER:
Be precise and very concise.

""")

rag_chain = (

{'context': lambda x: retriever.invoke(x['question']),
 'question': lambda x: x['question']}

|prompt
|llm
|parser


)



In [6]:
question = 'what is the sector analysis tries to figure out?'

question

'what is the sector analysis tries to figure out?'

In [7]:
response = rag_chain.invoke({'question':question})

In [8]:
print(f'USER_Query : {question}\n')

print('*'*50, '\n')

print(f'RAG_Answer : \n  {response}' )



USER_Query : what is the sector analysis tries to figure out?

************************************************** 

RAG_Answer : 
  Sector analysis tries to figure out which sectors require immediate retention and service intervention, and which should be defended and expanded as part of a long-term growth strategy.
